In [ ]:
# ==============================================================================
# PHASE 0: Preparation - change working directory, load packages
# ==============================================================================
#cd ~/Desktop/retail_dashboard

# 1. Standard Library Modules
import os
import re
from pathlib import Path
from datetime import datetime

# 2. Data Manipulation & Numerical Computation
import numpy as np
import pandas as pd

# 3. Interactive Visualization Engine
import plotly.express as px
import plotly.graph_objects as go

# 4. Web Application Framework
import streamlit as st
# Health check confirmation
print(f"Pandas version:    {pd.__version__}")
print(f"NumPy version:     {np.__version__}")
print(f"Streamlit version: {st.__version__}")


# ==============================================================================
# PHASE 1: Data cleaning
# ==============================================================================

# ==============================================================================
### data cleaning - customers
# ==============================================================================

# Ingest and create an isolated working copy (preserve the raw data)
customers_raw = pd.read_csv("data/customers.csv")

# 1. Inspect dimensions and memory usage
print(f"Shape: {customers_raw.shape[0]} rows, {customers_raw.shape[1]} columns")
# 2. View distributions of numeric columns
display(customers_raw.describe().T)

# 3. Check null counts per column sorted descending
null_counts = customers_raw.isna().sum()
display(null_counts[null_counts > 0].sort_values(ascending=False))

#Step 2: Normalize Column Headers
customers_raw.columns = (
    customers_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
    .str.replace(r'[^\w_]', '', regex=True)
)

print("Standardized headers:", list(customers_raw.columns))

#Step 3: Handle Duplicates - Total exact row duplicates
print(f"Exact duplicates: {customers_raw.duplicated().sum()}")
customers_clean = customers_raw.drop_duplicates(subset=['customer_id'], keep='last')

#Step 4: Fix Types & Clean Strings
# 4.1. Strip whitespace from all object/string columns
customers_string_cols = customers_clean.select_dtypes(include=['object', 'string']).columns
for col in customers_string_cols:
    customers_clean[col] = customers_clean[col].astype(str).str.strip()

### Step 5: Address Missing Values & Outliers
# 5.1. Drop records where essential unique identifiers are missing
customers_clean = customers_clean.dropna(subset=['customer_id'])

# missing values in Customers.csv
customers_clean.isna().sum().sort_values(ascending = False)
customers_clean['gender']

### replacing NA with "unknown"
customers_clean['gender'] = customers_clean['gender'].fillna("Unknown")
customers_clean['age'] = customers_clean['age'].fillna(0)
customers_clean['phone'] = customers_clean['phone'].fillna('0910000001')
customers_clean['email'] = customers_clean['email'].fillna('default@gmail.com')
customers_clean['loyalty_tier'] = customers_clean['loyalty_tier'].fillna('Unknown')


### mixed gender level - add a new gender_clean column
customers_clean["gender"] = (
    customers_clean["gender"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "m": "Male",
        "male": "Male",
        "M": "Male",
        "f": "Female",
        "F": "Female",
        "female": "Female",
        "unknown": "Unknown"
    })
    .fillna("Unknown")
)
#check unique
customers_clean['gender'].unique()


# email cleaning
customers_clean['email'].head

# Strip surrounding whitespace and convert to lowercase
customers_clean['email'] = customers_clean['email'].str.strip().str.lower()

#Fix text substitutions like " at " or "[at]" to "@"
customers_clean['email'] = customers_clean['email'].str.replace(
    r'\s*(\[at\]|\bat\b)\s*', '@', regex=True
)

#Fix missing '@' before common domains (e.g., 'johnnygmail.com' -> 'johnny@gmail.com')
domains = ['gmail.com', 'yahoo.com', 'outlook.com', 'hotmail.com', 'icloud.com']
domain_regex = r'(?<!@)(' + '|'.join(map(re.escape, domains)) + r')$'

customers_clean['email'] = customers_clean['email'].str.replace(
    domain_regex, r'@\1', regex=True
)

#Download the cleaned costumers.csv
customers_clean.to_csv('customers_clean.csv', index=False)

# ==============================================================================
### data cleaning - products
# ==============================================================================

# Ingest and create an isolated working copy (preserve the raw data)
products_raw = pd.read_csv("data/products.csv")

# 1. Inspect dimensions and memory usage
print(f"Shape: {products_raw.shape[0]} rows, {products_raw.shape[1]} columns")

# 2. View distributions of numeric columns
display(products_raw.describe().T)

# 3. Check null counts per column sorted descending
null_counts = products_raw.isna().sum()
display(null_counts[null_counts > 0].sort_values(ascending=False))

#Step 2: Normalize Column Headers
products_raw.columns = (
    products_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
    .str.replace(r'[^\w_]', '', regex=True)
)

print("Standardized headers:", list(products_raw.columns))

#Step 3: Handle Duplicates
# 3.1. Total exact row duplicates
print(f"Exact duplicates: {products_raw.duplicated().sum()}") #no duplicates

# create a clean data
products_clean = products_raw.copy()

#Step 4: Fix Types & Clean Strings
# 4.1. Strip whitespace from all object/string columns
products_string_cols = products_clean.select_dtypes(include=['object', 'string']).columns
for col in products_string_cols:
    products_clean[col] = products_clean[col].astype(str).str.strip()

# 4.2. Convert date strings to timezone-aware UTC datetime
products_clean["launch_date"] = pd.to_datetime(
    products_clean["launch_date"],
    format="mixed",
    errors="coerce"
)

# 4.3. remove pesos sign
def clean_currency(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace("₱", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

# clean the data by removing Peso sign
products_clean['unit_price'] = clean_currency(
    products_clean['unit_price']
)

products_clean['unit_cost'] = clean_currency(
    products_clean['unit_cost']
)

# Step 5: Address Missing Values & Outliers
# Shows the exact count of missing values per column
products_clean.isna().sum()
# Fill missing stock with 0 and convert to integer
products_clean['stock_qty'] = products_clean['stock_qty'].fillna(0).astype(int)
# Strip leading and trailing whitespace from string entries
products_clean['category'] = products_clean['category'].str.strip()

# Mapping dictionary: dirty value -> standard value
category_map = {
    'Fashon': 'Fashion',
    'electronics': 'Electronics',
    'Elec': 'Electronics',
    'Toy': 'Toys',
    'Book': 'Books',
    'Grocery': 'Groceries',
    'Beauty & Care': 'Beauty',
    'Home & Kitchen': 'Home and Kitchen',
    'Sports & Outdoor': 'Sports'
}

# Apply the mapping
products_clean['category'] = products_clean['category'].replace(category_map)
products_clean['category'].unique()

#Download the cleaned products as cvs
products_clean.to_csv('products_clean.csv', index=False)


# ==============================================================================
### data cleaning - orders
# ==============================================================================

# Ingest and create an isolated working copy (preserve the raw data)
orders_raw = pd.read_csv("data/orders.csv")

# 1. Inspect dimensions and memory usage
print(f"Shape: {orders_raw.shape[0]} rows, {orders_raw.shape[1]} columns")

# 2. View distributions of numeric columns
display(orders_raw.describe().T)

# 3. Check null counts per column sorted descending
null_counts = orders_raw.isna().sum()
display(null_counts[null_counts > 0].sort_values(ascending=False))

#Step 2: Normalize Column Headers
orders_raw.columns = (
    orders_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
    .str.replace(r'[^\w_]', '', regex=True)
)

print("Standardized headers:", list(orders_raw.columns))

#Step 3: Handle Duplicates and create clean data
# 3.1. Total exact row duplicates
print(f"Exact duplicates: {orders_raw.duplicated().sum()}")

orders_clean = orders_raw.copy()

# 3.2. Deduplicate on business identifier (keep the most recent timestamp)
# Sort ascending, then drop duplicates keeping the last (newest) record
orders_clean = orders_clean.drop_duplicates(subset=['order_id'], keep='last')

##Step 4: Fix Types & Clean Strings
# 4.1. Strip whitespace from all object/string columns
orders_string_cols = orders_clean.select_dtypes(include=['object', 'string']).columns
for col in orders_string_cols:
    orders_clean[col] = orders_clean[col].astype(str).str.strip()

# 4.2. Convert date strings to timezone-aware UTC datetime
## mixed date
orders_clean.head(15)

orders_clean["order_datetime"] = pd.to_datetime(
    orders_clean["order_datetime"],
    format="mixed",
    errors="coerce"
)

# missing discount - replace NAN with zero it means zero discount or zero shipping fee
orders_clean['discount_pct'] = orders_clean['discount_pct'].fillna(0)
orders_clean['shipping_fee'] = orders_clean['shipping_fee'].fillna(0)

### final check
orders_clean['sales_channel'].unique()
orders_clean['order_status'].unique()
orders_clean['shipping_method'].unique()
display(orders_clean.describe().T)


#6 Download the cleaned orders as cvs
orders_clean.to_csv('orders_clean.csv', index=False)

# ==============================================================================
### data cleaning - items
# ==============================================================================

# Ingest and create an isolated working copy (preserve the raw data)
items_raw = pd.read_csv("data/order_items.csv")

# 1. Inspect dimensions and memory usage
print(f"Shape: {items_raw.shape[0]} rows, {items_raw.shape[1]} columns")

# 2. View distributions of numeric columns
display(items_raw.describe().T)

# 3. Check null counts per column sorted descending
null_counts = items_raw.isna().sum()
display(null_counts[null_counts > 0].sort_values(ascending=False))

#Step 2: Normalize Column Headers
items_raw.columns = (
    items_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
    .str.replace(r'[^\w_]', '', regex=True)
)

print("Standardized headers:", list(items_raw.columns))

#Step 3: Handle Duplicates
# 3.1. Total exact row duplicates
print(f"Exact duplicates: {items_raw.duplicated().sum()}") #262 duplicates

# Sort ascending, then drop duplicates keeping the last (newest) record
items_clean = items_raw.drop_duplicates(subset=['order_item_id'], keep='last')

##Step 4: Fix Types & Clean Strings
# 4.1 Strip whitespace from all object/string columns
items_string_cols = items_clean.select_dtypes(include=['object', 'string']).columns
for col in items_string_cols:
    items_clean[col] = items_clean[col].astype(str).str.strip()


#4.2 Remove Peso sign
def clean_currency(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace("₱", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )
### clean the data by removing Peso sign
items_clean['unit_price_at_purchase'] = clean_currency(
    items_clean['unit_price_at_purchase']
)

# Step 5: Address Missing Values & Outliers
# Shows the exact count of missing values per column
items_clean.isna().sum()

# 5.1. Fill missing price with the median historical purchase price of that specific product
items_clean['unit_price_at_purchase'] = items_clean.groupby('product_id')['unit_price_at_purchase'].transform(
    lambda group: group.fillna(group.median())
)

# 5.2. Fallback: If a product ID has no price history at all, fill with the global median
items_clean['unit_price_at_purchase'] = items_clean['unit_price_at_purchase'].fillna(
    items_clean['unit_price_at_purchase'].median()
)

# 5.3. Round to 2 decimal places for currency consistency
items_clean['unit_price_at_purchase'] = items_clean['unit_price_at_purchase'].round(2)
items_clean.isna().sum()

#6 Download the cleaned items as cvs
items_clean.to_csv('items_clean.csv', index=False)



# ==============================================================================
### data cleaning - payments
# ==============================================================================

# Ingest and create an isolated working copy (preserve the raw data)
payments_raw = pd.read_csv("data/payments.csv")

# 1. Inspect dimensions and memory usage
print(f"Shape: {payments_raw.shape[0]} rows, {payments_raw.shape[1]} columns")

# 2. View distributions of numeric columns
display(payments_raw.describe().T)

# 3. Check null counts per column sorted descending
null_counts = payments_raw.isna().sum()
display(null_counts[null_counts > 0].sort_values(ascending=False))

#Step 2: Normalize Column Headers
payments_raw.columns = (
    payments_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
    .str.replace(r'[^\w_]', '', regex=True)
)

print("Standardized headers:", list(payments_raw.columns))

#Step 3: Handle Duplicates
# 3.1. Total exact row duplicates
print(f"Exact duplicates: {payments_raw.duplicated().sum()}") #no duplicates

#3.2 create a clean dataset
payments_clean = payments_raw.copy()

##Step 4: Fix Types & Clean Strings
# 4.1. Strip whitespace from all object/string columns
payments_string_cols = payments_clean.select_dtypes(include=['object', 'string']).columns
for col in payments_string_cols:
    payments_clean[col] = payments_clean[col].astype(str).str.strip()

# 4.2. Convert date strings to timezone-aware UTC datetime
## mixed date

payments_clean["payment_date"] = pd.to_datetime(
    payments_clean["payment_date"],
    format="mixed",
    errors="coerce"
)

# 4.3. remove Peso sign
def clean_currency(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace("₱", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )
### clean the data by removing Peso sign
payments_clean['amount_paid'] = clean_currency(
    payments_clean['amount_paid']
)

### Step 5: Address, Zeroes, Missing Values & Outliers

# Create amount_paid_clean as numeric float
payments_clean['amount_paid_clean'] = pd.to_numeric(
    payments_clean['amount_paid'], 
    errors='coerce'
)

# Convert Paid transactions with 0.0 to NaN
paid_zero_mask = (payments_clean['payment_status'] == 'Paid') & (payments_clean['amount_paid_clean'] == 0.0)
payments_clean.loc[paid_zero_mask, 'amount_paid_clean'] = np.nan

# Ensure all missing values for Pending, Failed, and Refunded are set to 0.0
non_paid_mask = payments_clean['payment_status'].isin(['Pending', 'Failed', 'Refunded'])
payments_clean.loc[non_paid_mask & payments_clean['amount_paid_clean'].isna(), 'amount_paid_clean'] = 0.0

# Calculate clean medians per payment method
true_paid_medians = (
    payments_clean[payments_clean['payment_status'] == 'Paid']
    .groupby('payment_method')['amount_paid_clean']
    .median()
)

# Impute all missing Paid rows (original NaNs + converted placeholder zeroes)
paid_missing_mask = (payments_clean['payment_status'] == 'Paid') & payments_clean['amount_paid_clean'].isna()
payments_clean.loc[paid_missing_mask, 'amount_paid_clean'] = (
    payments_clean.loc[paid_missing_mask, 'payment_method'].map(true_paid_medians)
)

# Round to 2 decimal places
payments_clean['amount_paid_clean'] = payments_clean['amount_paid_clean'].round(2)


# Check for remaining nulls
print("Remaining NaNs:", payments_clean['amount_paid_clean'].isna().sum())

# Check breakdown of remaining zeroes
print("\nZeroes by status:")
print(payments_clean[payments_clean['amount_paid_clean'] == 0]['payment_status'].value_counts())


display(payments_clean.describe().T)
payments_clean['amount_paid_clean'].min()
payments_clean['amount_paid_clean'].max()
(payments_clean['amount_paid_clean'] == 0).sum()

#6 Download the cleaned payments as cvs
payments_clean.to_csv('payments_clean.csv', index=False)


# ==============================================================================
### data cleaning - shipments.csv
# ==============================================================================

# Ingest and create an isolated working copy (preserve the raw data)
shipments_raw = pd.read_csv("data/shipments.csv")

# 1.1. Inspect dimensions and memory usage
print(f"Shape: {shipments_raw.shape[0]} rows, {shipments_raw.shape[1]} columns")

# 1.2. View distributions of numeric columns
display(shipments_raw.describe().T)

# 1.3. Check null counts per column sorted descending
null_counts = shipments_raw.isna().sum()
display(null_counts[null_counts > 0].sort_values(ascending=False))

#Step 2: Normalize Column Headers
shipments_raw.columns = (
    shipments_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
    .str.replace(r'[^\w_]', '', regex=True)
)

print("Standardized headers:", list(shipments_raw.columns))

#Step 3: Handle Duplicates
# 3.1. Total exact row duplicates
print(f"Exact duplicates: {shipments_raw.duplicated().sum()}") #no duplicates

# 3.2 create a clean dataset
shipments_clean = shipments_raw.copy()

##Step 4: Fix Types & Clean Strings
# 4.1. Strip whitespace from all object/string columns
shipments_string_cols = shipments_clean.select_dtypes(include=['object', 'string']).columns
for col in shipments_string_cols:
    shipments_clean[col] = shipments_clean[col].astype(str).str.strip()

# 4.2. Convert date strings to timezone-aware UTC datetime
## mixed date

shipments_clean["ship_date"] = pd.to_datetime(
    shipments_clean["ship_date"],
    format="mixed",
    errors="coerce"
)

shipments_clean["delivered_date"] = pd.to_datetime(
    shipments_clean["delivered_date"],
    format="mixed",
    errors="coerce"
)


### Step 5: Address Missing Values & Outliers
# Shows the exact count of missing values per column
shipments_clean.isna().sum()  #729 missing values in delivered_date

# Crosstab comparing shipment_status against missing delivered_date
status_missing_ct = pd.crosstab(
    shipments_clean['shipment_status'],
    shipments_clean['delivered_date'].isna(),
    rownames=['Shipment Status'],
    colnames=['Is Delivered Date Missing?']
)

print(status_missing_ct)


# Group by shipment status and sum/mean the boolean isna() mask
missing_summary = shipments_clean.groupby('shipment_status')['delivered_date'].agg(
    total_records='count',                          # counts non-nulls
    missing_count=lambda s: s.isna().sum(),
    missing_pct=lambda s: (s.isna().mean() * 100).round(2)
)

print(missing_summary)

# 5.1. Nullify delivered_date for non-delivered statuses
non_delivered_mask = shipments_clean['shipment_status'].isin(
    ['In Transit', 'Lost', 'Returned to Sender']
)
shipments_clean.loc[non_delivered_mask, 'delivered_date'] = pd.NaT

# 2. Identify legitimate completed deliveries that already have valid dates
valid_delivered_mask = (shipments_clean['shipment_status'] == 'Delivered') & (
    shipments_clean['delivered_date'].notna()
)

# 3. Calculate transit duration (in days) using only confirmed deliveries
transit_days = (
    shipments_clean.loc[valid_delivered_mask, 'delivered_date']
    - shipments_clean.loc[valid_delivered_mask, 'ship_date']
).dt.days

# 4. Compute median transit days per courier
courier_medians = (
    shipments_clean.loc[valid_delivered_mask]
    .assign(transit_days=transit_days)
    .groupby('courier')['transit_days']
    .median()
)

# 5. Locate missing Delivered records and impute: ship_date + courier median transit days
missing_delivered_mask = (
    shipments_clean['shipment_status'] == 'Delivered'
) & (shipments_clean['delivered_date'].isna())

imputed_offsets = shipments_clean.loc[missing_delivered_mask, 'courier'].map(
    courier_medians
)
shipments_clean.loc[missing_delivered_mask, 'delivered_date'] = shipments_clean[
    'ship_date'
] + pd.to_timedelta(imputed_offsets, unit='D')


#6 Download the cleaned shipments as cvs
shipments_clean.to_csv('shipments_clean.csv', index=False)


# ==============================================================================
### data cleaning - returns.csv
# ==============================================================================

# Ingest and create an isolated working copy (preserve the raw data)
returns_raw = pd.read_csv("data/returns.csv")

# 1. Inspect dimensions and memory usage
print(f"Shape: {returns_raw.shape[0]} rows, {returns_raw.shape[1]} columns")

# 2. View distributions of numeric columns
display(returns_raw.describe().T)

# 3. Check null counts per column sorted descending
null_counts = returns_raw.isna().sum()
display(null_counts[null_counts > 0].sort_values(ascending=False))

#Step 2: Normalize Column Headers
returns_raw.columns = (
    returns_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
    .str.replace(r'[^\w_]', '', regex=True)
)

print("Standardized headers:", list(returns_raw.columns))

#Step 3: Handle Duplicates
# 3.1. Total exact row duplicates
print(f"Exact duplicates: {returns_raw.duplicated().sum()}") #no duplicates

#3.2 create a clean dataset
returns_clean = returns_raw.copy()

##Step 4: Fix Types & Clean Strings
# 4.1. Strip whitespace from all object/string columns
returns_string_cols = returns_clean.select_dtypes(include=['object', 'string']).columns
for col in returns_string_cols:
    returns_clean[col] = returns_clean[col].astype(str).str.strip()

# 4.2. Convert date strings to timezone-aware UTC datetime
## mixed date

returns_clean["return_request_date"] = pd.to_datetime(
    returns_clean["return_request_date"],
    format="mixed",
    errors="coerce"
)

# Cross-tabulate flag against whether refund amount is missing
print(
    pd.crosstab(
        returns_clean['refunded_flag'].fillna('MISSING'),
        returns_clean['refund_amount'].isna(),
        rownames=['Refunded Flag'],
        colnames=['Is Refund Amount Missing?'],
    )
)

### Step 5: Address Missing Values & Outliers
# Shows the exact count of missing values per column
returns_clean.isna().sum()

#step 5.1 create refunded_flag_clean
#Initialize from raw column
returns_clean['refunded_flag_clean'] = returns_clean['refunded_flag']

#Reconcile missing flags based on whether money was paid out
has_positive_refund = returns_clean['refund_amount'] > 0
missing_flag = returns_clean['refunded_flag_clean'].isna()

# If money was refunded but flag was missing -> 'Yes'
returns_clean.loc[missing_flag & has_positive_refund, 'refunded_flag_clean'] = 'Yes'

# Remaining missing flags (no money refunded) -> 'No'
returns_clean['refunded_flag_clean'] = returns_clean['refunded_flag_clean'].fillna('No')


#5.2 Create and impute
# 1. Initialize as numeric float
returns_clean['refund_amount_clean'] = pd.to_numeric(
    returns_clean['refund_amount'], 
    errors='coerce'
)

# 2. Rule: If not refunded ('No'), the cash outflow is strictly 0.0
returns_clean.loc[
    returns_clean['refunded_flag_clean'] == 'No', 
    'refund_amount_clean'
] = 0.0

# 3. Calculate median positive refund for 'Yes' and 'Partial'
positive_refunds = returns_clean[
    (returns_clean['refunded_flag_clean'].isin(['Yes', 'Partial'])) & 
    (returns_clean['refund_amount_clean'] > 0)
]
refund_medians = positive_refunds.groupby('refunded_flag_clean')['refund_amount_clean'].median()

# 4. Impute missing amounts for 'Yes' and 'Partial' with their corresponding medians
missing_refund_mask = (
    returns_clean['refunded_flag_clean'].isin(['Yes', 'Partial'])
) & (returns_clean['refund_amount_clean'].isna())

returns_clean.loc[missing_refund_mask, 'refund_amount_clean'] = (
    returns_clean.loc[missing_refund_mask, 'refunded_flag_clean'].map(refund_medians)
)

# 5. Round to standard 2 decimal places
returns_clean['refund_amount_clean'] = returns_clean['refund_amount_clean'].round(2)

# Check for nulls
print("Missing counts in clean columns:")
print(returns_clean[['refunded_flag_clean', 'refund_amount_clean']].isna().sum())


#6 Download the cleaned returns as cvs
returns_clean.to_csv('returns_clean.csv', index=False)


# ==============================================================================
# PHASE 2: Data Analysis
# ==============================================================================


#2.1 creating the fact table - 
# capstone task #3) Create a single 'fact table' for analysis (orders + items + products + customers).


import numpy as np
import pandas as pd

print(products_clean.columns.tolist())
print(orders_clean.columns.tolist())
print(customers_clean.columns.tolist())

# 1. Prepare dimension slices with clean column names
products_dim = products_clean[[
    'product_id',
    'product_name',
    'brand',
    'category',
    'unit_cost',
    'unit_price',
]]

orders_dim = orders_clean[[
    'order_id',
    'customer_id',
    'order_datetime',
    'order_status',
    'sales_channel',
    'shipping_method',
    'discount_pct',
    'shipping_fee',
]]

customers_dim = customers_clean[[
    'customer_id',
    'first_name',
    'last_name',
    'gender',
    'age',
    'region_name',
    'city',
    'loyalty_tier',
]]


# Step 2: Assemble the Fact Table (fact_sales)
#Join the dimensional slices sequentially onto order_items_clean:

# 1. Join Order Items with Products
fact_sales = pd.merge(
    items_clean,
    products_dim,
    on='product_id',
    how='left',
    suffixes=('', '_catalog'),
)

# 2. Join with Orders
fact_sales = pd.merge(fact_sales, orders_dim, on='order_id', how='left')

# 3. Join with Customers
fact_sales = pd.merge(fact_sales, customers_dim, on='customer_id', how='left')


# 3. Time-Series Attributes (Ensure datetime dtype)
fact_sales['order_datetime'] = pd.to_datetime(
    fact_sales['order_datetime'], format='mixed'
)
fact_sales['order_date'] = fact_sales['order_datetime'].dt.date
fact_sales['year_month'] = fact_sales['order_datetime'].dt.to_period('M')

# 4. Feature Engineering: Line-Item Financials
fact_sales['gross_revenue'] = (
    fact_sales['quantity'] * fact_sales['unit_price_at_purchase']
).round(2)

fact_sales['discount_amount'] = (
    fact_sales['gross_revenue'] * fact_sales['discount_pct'].fillna(0)
).round(2)

fact_sales['net_revenue'] = (
    fact_sales['gross_revenue'] - fact_sales['discount_amount']
).round(2)

fact_sales['cogs'] = (fact_sales['quantity'] * fact_sales['unit_cost']).round(2)

fact_sales['gross_profit'] = (
    fact_sales['net_revenue'] - fact_sales['cogs']
).round(2)

#Step 4: Verification & Sanity Audit
print(f"Fact Table Shape: {fact_sales.shape}")
print(f"Total Unique Orders: {fact_sales['order_id'].nunique():,}")
print(f"Total Line Items: {len(fact_sales):,}")
print(f"Total Net Revenue: ₱{fact_sales['net_revenue'].sum():,.2f}")



# Phase 2.2 KPI
# capstone  #4) Compute KPIs:
#- total revenue, total orders, average order value (AOV)
#- monthly revenue trend
#- top 10 products by revenue
#- revenue by category
#- revenue by region/city
#- repeat vs new customers (simple definition allowed)



# 1. Total Net Revenue and Total Orders
total_net_revenue = fact_sales['net_revenue'].sum()
total_orders = fact_sales['order_id'].nunique()

# 2. Average Order Value (AOV)
order_level_revenue = fact_sales.groupby('order_id')['net_revenue'].sum()
aov = order_level_revenue.mean()

print(f"Total Net Revenue:  ₱{total_net_revenue:,.2f}")
print(f"Total Unique Orders: {total_orders:,}")
print(f"Average Order Value: ₱{aov:,.2f}")


#Step 2: Monthly Revenue Trend
# Aggregate net revenue by year-month period
monthly_trend = (
    fact_sales.groupby('year_month')['net_revenue']
    .sum()
    .reset_index()
    .rename(columns={'net_revenue': 'monthly_revenue'})
)

# Convert Period to string for plotting libraries (Streamlit / Plotly)
monthly_trend['month_str'] = monthly_trend['year_month'].astype(str)
print(monthly_trend.head(6))

#Step 3: Top 10 Products by Revenue
top_10_products = (
    fact_sales.groupby(['product_id', 'product_name'])['net_revenue']
    .sum()
    .reset_index()
    .sort_values(by='net_revenue', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print(top_10_products)

#Step 4: Revenue by Category & Category Sanitation Check
revenue_by_category = (
    fact_sales.groupby('category')['net_revenue']
    .sum()
    .reset_index()
    .sort_values(by='net_revenue', ascending=False)
)

print(revenue_by_category)

#Step 5: Revenue by Region and City
# 1. By Region
revenue_by_region = (
    fact_sales.groupby('region_name')['net_revenue']
    .sum()
    .reset_index()
    .sort_values(by='net_revenue', ascending=False)
)

# 2. By City
revenue_by_city = (
    fact_sales.groupby(['region_name', 'city'])['net_revenue']
    .sum()
    .reset_index()
    .sort_values(by='net_revenue', ascending=False)
)

#print("Top 5 Regions by Net Revenue:")
print(revenue_by_region.head(5))
print(revenue_by_city.head(5))

#Step 6: Repeat vs. New Customers
# Count orders per customer across the dataset
customer_orders = fact_sales.groupby('customer_id')['order_id'].nunique()

# Classify customer retention
repeat_customers_count = (customer_orders > 1).sum()
new_customers_count = (customer_orders == 1).sum()
total_customers = len(customer_orders)

repeat_pct = (repeat_customers_count / total_customers) * 100
new_pct = (new_customers_count / total_customers) * 100

customer_retention_df = pd.DataFrame({
    'Customer Type': ['Repeat Customers (2+ orders)', 'Single-Order Customers (1 order)'],
    'Customer Count': [repeat_customers_count, new_customers_count],
    'Percentage': [f"{repeat_pct:.1f}%", f"{new_pct:.1f}%"]
})

print(customer_retention_df)


# ==============================================================================
# PHASE 3: EDA and outlier
# ==============================================================================
target_col = "category_clean" if "category_clean" in fact_sales.columns else "category"

unique_cats = fact_sales[target_col].unique()
print(f"Total Unique Categories in '{target_col}':", len(unique_cats))
print("\nUnique Category List:")
print(unique_cats)

# Handle missing category from orphan product records (e.g., PROD00000)
fact_sales['category'] = fact_sales['category'].fillna('Unknown / Uncategorized')
print(fact_sales['category'].value_counts(dropna=False))


#Phase 3, Step 3.2: Distribution & Outlier Analysis
# 1. Profile key financial & operational metrics at line-item level
distribution_cols = ['quantity', 'unit_price_at_purchase', 'discount_pct', 'net_revenue']
print("--- Line-Item Distribution Summary ---")
print(fact_sales[distribution_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2))

# 2. Profile basket sizes and order-level totals
order_totals = fact_sales.groupby('order_id').agg(
    order_items_count=('order_item_id', 'count'),
    total_quantity=('quantity', 'sum'),
    order_net_revenue=('net_revenue', 'sum')
)

print("\n--- Order-Level Distribution Summary ---")
print(order_totals.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2))

### Step 3.4: Analytical Artifact Serialization (Export to Parquet)
# Convert Period and object dates to strings/standard datetimes before export
fact_sales_export = fact_sales.copy()
fact_sales_export['year_month'] = fact_sales_export['year_month'].astype(str)
fact_sales_export['order_date'] = pd.to_datetime(fact_sales_export['order_date'])

# Export optimized analytical model
fact_sales_export.to_parquet('fact_sales.parquet', index=False)

csv_path = "streamlit/clean_data/fact_sales.csv"
fact_sales.to_csv(csv_path, index=False)
##########

# ==============================================================================
# PHASE 4: Dashboard via Streamlit
# ==============================================================================
